# 04 — TabNet kurulum ve ilk deneme

**Sahibi:** Öğrenci 3 (TabNet & Comparative Analytics)

Bu notebook'un bu bölümü plan madde 9 / Adım 1-2'yi karşılar:
- `torch` ve `pytorch_tabnet` kurulumunun çalıştığını doğrula.
- 2.000 satırlık bir örneklemle 5 epoch'luk hızlı bir deneme yap.
- Kategorik kolonları `CAT_LEVELS`'tan türetilen sabit kodlarla 0..k-1 aralığına eşle
  (veriden değil, sözleşmeden türetilir ki test setinde görülmemiş bir kod çökme yaratmasın).


In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Adım 1: Kurulum kontrolü

In [2]:
import torch
import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
from importlib.metadata import version

print("torch         ", torch.__version__)
print("pytorch_tabnet", version("pytorch-tabnet"))
print("cuda var mı   ", torch.cuda.is_available(), "(CPU sürümüyle False beklenir)")

torch          2.14.1+cpu
pytorch_tabnet 4.1.0
cuda var mı    False (CPU sürümüyle False beklenir)


## Adım 2: Kategorik hazırlık

`cat_idxs` ve `cat_dims`, veriden değil `CAT_LEVELS` sözleşmesinden hesaplanır.
Sayısal kolonlar `make_preprocessor("tabnet")` ile ölçeklenir, kategorikler bu
hücrede 0..k-1 aralığına eşlenir (`make_preprocessor("tabnet")` kodları ham
bırakır, TabNet embedding'i 0'dan başlayan ardışık kod bekler).

In [3]:
from src.config import CAT_COLS, CAT_LEVELS, FEATURES, NUM_COLS
from src.data import load_split, to_xy
from src.features import make_preprocessor

CODE_MAPS = {col: {code: i for i, code in enumerate(levels)} for col, levels in CAT_LEVELS.items()}


def encode_categoricals(X):
    """Map each categorical column's raw codes to 0..k-1 using CAT_LEVELS."""
    X = X.copy()
    for col in CAT_COLS:
        X[col] = X[col].map(CODE_MAPS[col]).astype("int64")
    return X


cat_idxs = [FEATURES.index(col) for col in CAT_COLS]
cat_dims = [len(CAT_LEVELS[col]) for col in CAT_COLS]
list(zip(CAT_COLS, cat_idxs, cat_dims))

[('SEX', 1, 2),
 ('EDUCATION', 2, 7),
 ('MARRIAGE', 3, 4),
 ('PAY_0', 5, 11),
 ('PAY_2', 6, 11),
 ('PAY_3', 7, 11),
 ('PAY_4', 8, 11),
 ('PAY_5', 9, 11),
 ('PAY_6', 10, 11)]

## Adım 3: 2.000 satırlık hızlı deneme (5 epoch)

In [4]:
from src.config import SEED

VARIANT = "v0"  # Öğrenci 1'in clean_* dosyaları gelince "clean" olacak

train_full = load_split("train", VARIANT)
sample = train_full.sample(n=2000, random_state=SEED)
X_sample, y_sample = to_xy(sample)

preproc = make_preprocessor("tabnet")
X_scaled = preproc.fit_transform(X_sample)
X_scaled = encode_categoricals(X_scaled)
X_scaled = X_scaled[FEATURES].to_numpy(dtype="float32")
y_arr = y_sample.to_numpy()

X_scaled.shape, y_arr.shape

((2000, 23), (2000,))

In [5]:
clf = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.3,
    seed=SEED,
    verbose=1,
)
clf.fit(
    X_scaled,
    y_arr,
    max_epochs=5,
    batch_size=256,
    virtual_batch_size=64,
)
print("Hızlı deneme tamamlandı, TabNet kurulumu çalışıyor.")

epoch 0  | loss: 0.64045 |  0:00:00s
epoch 1  | loss: 0.55545 |  0:00:00s


epoch 2  | loss: 0.52433 |  0:00:00s
epoch 3  | loss: 0.51859 |  0:00:00s


epoch 4  | loss: 0.49852 |  0:00:00s
Hızlı deneme tamamlandı, TabNet kurulumu çalışıyor.
